# Final study: all four algorithms together

Runs **on your computer** (not on Kaggle): it reads the downloaded outputs of the two final notebooks
(A = MACURA + MBPO on account nouradon, B = M2AC + SAC on account silvergjeka01), merges them and makes the results:
the tables, the learning curves and scores, MACURA's trust in its world model, the world-model check on real flights,
and the race videos.

**How to run** (from the repo folder, once `./finals.sh get` has downloaded both outputs):

    pip install -r requirements.txt torch notebook       # once
    jupyter notebook notebooks/final_merge.ipynb          # then: Run All
    # or without opening it:  jupyter nbconvert --to notebook --execute --inplace notebooks/final_merge.ipynb

Everything is written to `results_final/` (tables in `summary.md`, figures in `plots/`, videos in `videos/`).

## 1. Which results to merge

In [ ]:
import os, sys, glob, json

ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# the downloaded outputs (folders next to the repo files); the first folder that has a run wins if one appears twice
RESULTS = ["out_final_a", "out_final_b"]
# if the race3 test (seeds 0-3, 50k, same settings) is used as notebook A instead of re-running it:
# RESULTS = ["out_race3_seeds01", "out_race3_seeds23", "out_final_b"]
PILOT   = "out_seed0_race2_pilot"   # the race2 pilot (seed 0, 50k), reported next to the final study
OUT_DIR = "results_final"
VIDEOS  = True                       # race videos (a few minutes)
EXPECTED_SEEDS = [0, 1, 2, 3]

def _first_log(folder):
    hits = sorted(glob.glob(os.path.join(folder, "**", "logs", "*_seed*.json"), recursive=True))
    return json.load(open(hits[0])) if hits else None

found = {f: _first_log(f) for f in RESULTS}
for f, run in found.items():
    print(f"{f:28s}", "NOT FOUND or empty" if run is None else f"ok (task {run.get('setup', {}).get('task')})")
TASK = next((r["setup"]["task"] for r in found.values() if r and r.get("setup", {}).get("task")), "race3")
os.environ["MACURA_TASK"] = TASK
os.environ.setdefault("MUJOCO_GL", "disable")      # plots here; videos are drawn in their own process
print("task:", TASK)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, Video, display
from src.config import config as cfg
from src.analysis import results as R, report
from src.viz import video as V

RUNS = R.load_runs(*[f for f in RESULTS if found.get(f)])
where = {(r["algo"], r["seed"]): next((f for f in RESULTS if os.path.abspath(r["_root"]).startswith(os.path.abspath(f))),
                                     r["_root"]) for r in RUNS}
WHERE_MD = ["| algorithm | " + " | ".join(f"seed {s}" for s in EXPECTED_SEEDS) + " |", "|---|" + "---|" * len(EXPECTED_SEEDS)]
for a in R.ALGOS:
    WHERE_MD.append(f"| {R.NAMES[a]} | " + " | ".join(where.get((a, s), "**missing**") for s in EXPECTED_SEEDS) + " |")
display(Markdown("### Which run comes from which folder\n\n" + "\n".join(WHERE_MD)))
missing = [f"{R.NAMES[a]} seed {s}" for a in R.ALGOS for s in EXPECTED_SEEDS if (a, s) not in where]
stopped = [f"{R.NAMES[r['algo']]} seed {r['seed']} (at step {r['setup']['stopped_early_at']})" for r in RUNS
           if r.get("setup", {}).get("stopped_early_at")]
print("missing runs:", ", ".join(missing) if missing else "none")
print("stopped early by the Kaggle time limit:", ", ".join(stopped) if stopped else "none")
STEPS = R.run_length(RUNS[0])
print(f"{len(RUNS)} runs, {STEPS:,} steps each")

## 2. Results
The scores fixed before the runs: **average return while learning** and **drones broken after the warm-up** (main),
then the best checkpoint's **test on 30 fresh scenarios** (return, success = % of a lap per 10 s flight, crash rate).
With 4 seeds: IQM over seeds with a 95% bootstrap confidence interval.

In [ ]:
display(Markdown(R.summary_markdown(RUNS)))
display(Markdown("### Per seed\n\n" + R.per_seed_markdown(RUNS)))

### MACURA against each method, seed by seed
The same seed gives every algorithm the same scenarios, wind and gusts, so the per-seed difference removes most of
the luck. "MACURA better in" counts the seeds where MACURA is better (higher return / success, fewer drones broken /
crashes).

In [ ]:
GAP = [("avg_return", "avg return while learning", "{:+.0f}", 1), ("broken", "drones broken after warm-up", "{:+.0f}", -1),
       ("test_return", "final test return", "{:+.0f}", 1), ("test_success", "success, % of a lap (test)", "{:+.0%}", 1),
       ("test_crash", "final test crash rate", "{:+.0%}", -1), ("last10_return", "return, last 10 evals", "{:+.0f}", 1)]
by = {(r["algo"], r["seed"]): R.run_scores(r) for r in RUNS}
GAP_MD = []
for other in ("mbpo", "m2ac", "sac"):
    seeds = [s for s in EXPECTED_SEEDS if ("macura", s) in by and (other, s) in by]
    if not seeds:
        continue
    rows = [f"| MACURA - {R.NAMES[other]} | " + " | ".join(f"seed {s}" for s in seeds) + " | mean | MACURA better in |",
            "|---|" + "---|" * (len(seeds) + 2)]
    for k, label, fmt, sign in GAP:
        d = [by[("macura", s)][k] - by[(other, s)][k] for s in seeds]
        rows.append(f"| {label} | " + " | ".join(fmt.format(x) for x in d) + f" | {fmt.format(np.mean(d))} | "
                    f"{sum(sign * x > 0 for x in d)} of {len(d)} |")
    GAP_MD.append(f"#### MACURA vs {R.NAMES[other]}\n\n" + "\n".join(rows))
    display(Markdown(GAP_MD[-1]))

### Fairness check and the chute
- **Updates per real step**: MBPO and M2AC were set to 12 = MACURA's average in the race2 pilot; this shows what each
  run actually did.
- **The chute**: did a *trained* policy lose lift? Only the second half of training and the final test count
  (lift loss starts at 1.2 m/s of sink); MACURA's trust in fast descents vs normal flight over the whole run.

In [ ]:
utd = ["| algorithm | " + " | ".join(f"seed {s}" for s in EXPECTED_SEEDS) + " |", "|---|" + "---|" * len(EXPECTED_SEEDS)]
for a in R.ALGOS:
    cells = []
    for s in EXPECTED_SEEDS:
        r = next((x for x in RUNS if x["algo"] == a and x["seed"] == s), None)
        cells.append("-" if r is None else (f"{np.mean(r['utd']):.1f}" if r.get("utd") else "1"))
    utd.append(f"| {R.NAMES[a]} | " + " | ".join(cells) + " |")
display(Markdown("#### Updates per real step (mean)\n\n" + "\n".join(utd)))

chute = ["| algorithm | seed | fastest descent, 2nd half (m/s) | losing lift, 2nd half | test: fastest descent | "
         "test: losing lift | MACURA trusts fast / normal |", "|---|---|---|---|---|---|---|"]
reached = []
for r in RUNS:
    late = np.asarray(r["steps"]) >= STEPS / 2
    sink, lift, fe = np.asarray(r["eval_max_sink"])[late], np.asarray(r["eval_liftloss"])[late], r["final_eval"]
    tr = "-"
    if r["algo"] == "macura" and r.get("trust_fast"):
        tr = f"{np.nanmean([v for _, v in r['trust_fast']]):.0%} / {np.nanmean([v for _, v in r['trust_slow']]):.0%}"
    if (len(lift) and lift.mean() > 0.01) or fe.get("eval_liftloss", 0) > 0.01:
        reached.append(f"{R.NAMES[r['algo']]} seed {r['seed']}")
    chute.append(f"| {R.NAMES[r['algo']]} | {r['seed']} | {sink.max() if len(sink) else float('nan'):.2f} | "
                 f"{100 * lift.mean() if len(lift) else float('nan'):.1f}% | {fe.get('eval_max_sink', float('nan')):.2f} | "
                 f"{100 * fe.get('eval_liftloss', 0):.1f}% | {tr} |")
display(Markdown("#### The chute\n\n" + "\n".join(chute)))
print("trained policies losing lift more than 1% of the time:", ", ".join(reached) if reached else "none")
print()
print(R.trust_summary(RUNS))

## 3. Figures and the world-model check
Learning curves (paper Fig. 4 style, IQM with 95% CI over the 4 seeds, the hand-written autopilot as reference lines),
the results table, the scores, how much each method trusts its world model, MACURA's threshold over training, the
imagined data above MACURA's threshold, and the **world-model check**: the best MACURA and MBPO policies fly 10 fresh
scenarios and, at every step, the model's disagreement is compared with its real one-step error (paper Fig. 10).
Takes a few minutes (it flies the autopilot and the policies).

In [ ]:
MADE = report.build(RUNS, cfg.CFG, OUT_DIR, check=True, videos=False, close=True)
for name in ("learning_curves", "summary_table", "scores", "model_trust", "kappa", "model_check", "untrusted_data"):
    if MADE.get(name):
        print(name); display(Image(MADE[name]))

## 4. Videos
The best seed of each algorithm (picked on the selection scenarios, never on the test ones) flies the **same three
fresh test scenarios** side by side (same start, package, wind and gusts); MACURA's panel shows its world model's
verdict at every step (green = members agree, orange = they disagree and MACURA would stop imagining). Then a 30 s
MACURA vs MBPO demo flight (longer than the 10 s training flights; not a score).

In [ ]:
if VIDEOS:
    best = {R.NAMES[a]: R.best_run(rs) for a, rs in R.by_algo(RUNS).items()}
    pilots = {k: R.checkpoint(r) for k, r in best.items()}
    trust = {k: {"ensemble": R.ensemble_checkpoint(r), "kappa": R.kappa_at(r)}
             for k, r in best.items() if r["algo"] == "macura" and R.ensemble_checkpoint(r)}
    print("best seed per algorithm:", {k: r["seed"] for k, r in best.items()})
    race = V.record_race(cfg.CFG, pilots, f"{OUT_DIR}/videos/race_all.mp4", seeds=(1000, 1001, 1002), trust=trust,
                         frame_step=3, fps=17)
    if race:
        display(Video(race, embed=True, width=960))
    duo = {k: v for k, v in pilots.items() if k in ("MACURA", "MBPO")}
    if len(duo) == 2:
        long = V.record_race(cfg.CFG, duo, f"{OUT_DIR}/videos/long_flight.mp4", seeds=(1010, 1011), trust=trust,
                             frame_step=3, fps=17, max_steps=1500)
        if long:
            display(Video(long, embed=True, width=960))

## 5. Next to the race2 pilot
The race2 pilot (seed 0, 50k steps, MBPO and M2AC with 8 updates per step) for comparison: a different task and a
different update budget, so it is context, not part of the final comparison.

In [ ]:
PILOT_RUNS = R.load_runs(PILOT, verbose=False) if os.path.isdir(PILOT) else []
if PILOT_RUNS:
    display(Markdown("#### race2 pilot, seed 0\n\n" + R.summary_markdown(PILOT_RUNS)))
else:
    print("race2 pilot not found at", PILOT)

## 6. Save

In [ ]:
parts = [open(MADE["summary"]).read(), "## MACURA against each method, seed by seed", *GAP_MD,
         "## Updates per real step", "\n".join(utd), "## The chute", "\n".join(chute),
         "## Which run comes from which folder", "\n".join(WHERE_MD)]
if PILOT_RUNS:
    parts += ["## race2 pilot (seed 0, 50k; context)", R.summary_markdown(PILOT_RUNS)]
open(f"{OUT_DIR}/summary.md", "w").write("\n\n".join(parts) + "\n")
print("saved:", os.path.abspath(OUT_DIR))
for p in sorted(glob.glob(f"{OUT_DIR}/**/*.*", recursive=True)):
    print("  ", os.path.relpath(p, OUT_DIR))